# Phase 1: Data Exploration and Profiling

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import Counter
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
DATASET_PATH = Path('Dataset/Animals-10')

classes = sorted([d.name for d in DATASET_PATH.iterdir() if d.is_dir()])
num_classes = len(classes)

print(f"Total number of classes: {num_classes}")
print(f"\nClass names:\n{', '.join(classes)}")

In [ ]:
class_counts = {}
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}

for class_name in classes:
    class_path = DATASET_PATH / class_name
    image_files = [f for f in class_path.iterdir() 
                   if f.suffix.lower() in image_extensions]
    class_counts[class_name] = len(image_files)

df_distribution = pd.DataFrame(list(class_counts.items()), 
                               columns=['Class', 'Count'])
df_distribution = df_distribution.sort_values('Count', ascending=False)

print("Distribution of images per class:")
print(df_distribution.to_string(index=False))
print(f"\nTotal images: {df_distribution['Count'].sum()}")
print(f"Average images per class: {df_distribution['Count'].mean():.2f}")
print(f"Median images per class: {df_distribution['Count'].median():.2f}")

In [ ]:
max_count = df_distribution['Count'].max()
min_count = df_distribution['Count'].min()
imbalance_ratio = max_count / min_count if min_count > 0 else 0

print(f"Class with most samples: {df_distribution.iloc[0]['Class']} ({max_count} images)")
print(f"Class with least samples: {df_distribution.iloc[-1]['Class']} ({min_count} images)")
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 2:
    print("\nSignificant class imbalance detected!")
else:
    print("\nClasses are relatively balanced")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
axes[0].bar(df_distribution['Class'], df_distribution['Count'], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Animal Class')
axes[0].set_ylabel('Number of Images')
axes[0].set_title('Image Distribution per Class')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

for i, (class_name, count) in enumerate(zip(df_distribution['Class'], df_distribution['Count'])):
    axes[0].text(i, count + max_count*0.01, str(count), ha='center', va='bottom', fontsize=9)

# Pie chart
colors = plt.cm.Set3(range(len(classes)))
axes[1].pie(df_distribution['Count'], labels=df_distribution['Class'], 
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proportion of Each Class')

plt.tight_layout()
plt.show()

In [ ]:
image_metadata = []

print("Analyzing image properties...")
for class_name in classes:
    class_path = DATASET_PATH / class_name
    image_files = [f for f in class_path.iterdir() 
                   if f.suffix.lower() in image_extensions]
    
    for img_path in image_files:
        try:
            with Image.open(img_path) as img:
                width, height = img.size
                mode = img.mode
                file_size = img_path.stat().st_size / 1024  # KB
                
                image_metadata.append({
                    'class': class_name,
                    'width': width,
                    'height': height,
                    'aspect_ratio': width / height,
                    'mode': mode,
                    'file_size_kb': file_size,
                    'format': img.format
                })
        except Exception as e:
            print(f"Error reading {img_path}: {e}")

df_metadata = pd.DataFrame(image_metadata)
print(f"Successfully analyzed {len(df_metadata)} images")

In [ ]:
print("=" * 60)
print("IMAGE DIMENSION STATISTICS")
print("=" * 60)
print(f"Width  - Mean: {df_metadata['width'].mean():.2f}, Std: {df_metadata['width'].std():.2f}")
print(f"         Min: {df_metadata['width'].min()}, Max: {df_metadata['width'].max()}")
print(f"\nHeight - Mean: {df_metadata['height'].mean():.2f}, Std: {df_metadata['height'].std():.2f}")
print(f"         Min: {df_metadata['height'].min()}, Max: {df_metadata['height'].max()}")
print(f"\nAspect Ratio - Mean: {df_metadata['aspect_ratio'].mean():.2f}")
print(f"               Range: [{df_metadata['aspect_ratio'].min():.2f}, {df_metadata['aspect_ratio'].max():.2f}]")

print("\n" + "=" * 60)
print("FILE PROPERTIES")
print("=" * 60)
print(f"File Size - Mean: {df_metadata['file_size_kb'].mean():.2f} KB")
print(f"            Min: {df_metadata['file_size_kb'].min():.2f} KB, Max: {df_metadata['file_size_kb'].max():.2f} KB")
print(f"\nTotal dataset size: {df_metadata['file_size_kb'].sum() / 1024:.2f} MB")

print("\n" + "=" * 60)
print("FORMAT AND MODE DISTRIBUTION")
print("=" * 60)
print(f"\nFormats:\n{df_metadata['format'].value_counts()}")
print(f"\nColor Modes:\n{df_metadata['mode'].value_counts()}")

In [ ]:
# Detect anomalies
q1_width = df_metadata['width'].quantile(0.25)
q3_width = df_metadata['width'].quantile(0.75)
iqr_width = q3_width - q1_width

q1_height = df_metadata['height'].quantile(0.25)
q3_height = df_metadata['height'].quantile(0.75)
iqr_height = q3_height - q1_height

outliers_width = df_metadata[(df_metadata['width'] < q1_width - 1.5*iqr_width) | 
                             (df_metadata['width'] > q3_width + 1.5*iqr_width)]
outliers_height = df_metadata[(df_metadata['height'] < q1_height - 1.5*iqr_height) | 
                              (df_metadata['height'] > q3_height + 1.5*iqr_height)]

print(f"Images with outlier widths: {len(outliers_width)}")
print(f"Images with outlier heights: {len(outliers_height)}")

# Check for non-RGB images
non_rgb = df_metadata[df_metadata['mode'] != 'RGB']
if len(non_rgb) > 0:
    print(f"\nFound {len(non_rgb)} non-RGB images:")
    print(non_rgb['mode'].value_counts())
else:
    print("\nAll images are in RGB mode")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Width distribution
axes[0, 0].hist(df_metadata['width'], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df_metadata['width'].mean(), color='red', linestyle='--', label=f"Mean: {df_metadata['width'].mean():.0f}")
axes[0, 0].set_xlabel('Width (pixels)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Image Widths')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Height distribution
axes[0, 1].hist(df_metadata['height'], bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(df_metadata['height'].mean(), color='red', linestyle='--', label=f"Mean: {df_metadata['height'].mean():.0f}")
axes[0, 1].set_xlabel('Height (pixels)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Image Heights')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Aspect ratio distribution
axes[1, 0].hist(df_metadata['aspect_ratio'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(1.0, color='red', linestyle='--', label='Square (1:1)')
axes[1, 0].set_xlabel('Aspect Ratio (Width/Height)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Aspect Ratios')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Width vs Height scatter
axes[1, 1].scatter(df_metadata['width'], df_metadata['height'], alpha=0.3, s=10, color='purple')
axes[1, 1].set_xlabel('Width (pixels)')
axes[1, 1].set_ylabel('Height (pixels)')
axes[1, 1].set_title('Width vs Height Scatter Plot')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for dimension comparison across classes
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

df_metadata.boxplot(column='width', by='class', ax=axes[0], rot=45)
axes[0].set_title('Width Distribution by Class')
axes[0].set_xlabel('Animal Class')
axes[0].set_ylabel('Width (pixels)')
axes[0].get_figure().suptitle('')

df_metadata.boxplot(column='height', by='class', ax=axes[1], rot=45)
axes[1].set_title('Height Distribution by Class')
axes[1].set_xlabel('Animal Class')
axes[1].set_ylabel('Height (pixels)')
axes[1].get_figure().suptitle('')

plt.tight_layout()
plt.show()

## Visual Content Analysis

In [ ]:
num_samples = 5
fig, axes = plt.subplots(num_classes, num_samples, figsize=(15, 20))

for i, class_name in enumerate(classes):
    class_path = DATASET_PATH / class_name
    image_files = [f for f in class_path.iterdir() 
                   if f.suffix.lower() in image_extensions]
    
    # Random sample
    samples = np.random.choice(image_files, size=min(num_samples, len(image_files)), replace=False)
    
    for j, img_path in enumerate(samples):
        try:
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            if j == 0:
                axes[i, j].set_title(class_name, fontsize=12, fontweight='bold', loc='left')
        except Exception as e:
            axes[i, j].text(0.5, 0.5, 'Error', ha='center', va='center')
            axes[i, j].axis('off')

plt.suptitle('Random Sample Images from Each Class', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
def analyze_image_properties(img_array):
    brightness = np.mean(img_array)
    contrast = np.std(img_array)
    
    if len(img_array.shape) == 3 and img_array.shape[2] == 3:
        r_mean = np.mean(img_array[:, :, 0])
        g_mean = np.mean(img_array[:, :, 1])
        b_mean = np.mean(img_array[:, :, 2])
        
        # Saturation approximation
        max_rgb = np.max(img_array, axis=2)
        min_rgb = np.min(img_array, axis=2)
        saturation = np.mean((max_rgb - min_rgb) / (max_rgb + 1e-7))
    else:
        r_mean = g_mean = b_mean = saturation = 0
    
    return brightness, contrast, r_mean, g_mean, b_mean, saturation

print("Analyzing visual properties...")
visual_properties = []

for class_name in classes:
    class_path = DATASET_PATH / class_name
    image_files = [f for f in class_path.iterdir() 
                   if f.suffix.lower() in image_extensions]
    
    # Sample subset for efficiency
    sample_size = min(100, len(image_files))
    sample_files = np.random.choice(image_files, size=sample_size, replace=False)
    
    for img_path in sample_files:
        try:
            with Image.open(img_path) as img:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                img_array = np.array(img)
                brightness, contrast, r, g, b, sat = analyze_image_properties(img_array)
                
                visual_properties.append({
                    'class': class_name,
                    'brightness': brightness,
                    'contrast': contrast,
                    'red_mean': r,
                    'green_mean': g,
                    'blue_mean': b,
                    'saturation': sat
                })
        except Exception as e:
            continue

df_visual = pd.DataFrame(visual_properties)
print(f"Analyzed {len(df_visual)} images for visual properties")

In [ ]:
visual_summary = df_visual.groupby('class').agg({
    'brightness': ['mean', 'std'],
    'contrast': ['mean', 'std'],
    'saturation': ['mean', 'std']
}).round(2)

print("Visual Properties Summary by Class:")
print(visual_summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Brightness distribution
axes[0, 0].hist(df_visual['brightness'], bins=50, color='gold', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Brightness (0-255)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Overall Brightness Distribution')
axes[0, 0].grid(alpha=0.3)

# Contrast distribution
axes[0, 1].hist(df_visual['contrast'], bins=50, color='orange', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Contrast (std)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Overall Contrast Distribution')
axes[0, 1].grid(alpha=0.3)

# Brightness by class
df_visual.boxplot(column='brightness', by='class', ax=axes[1, 0], rot=45)
axes[1, 0].set_title('Brightness by Class')
axes[1, 0].set_xlabel('Animal Class')
axes[1, 0].set_ylabel('Brightness')
axes[1, 0].get_figure().suptitle('')

# Contrast by class
df_visual.boxplot(column='contrast', by='class', ax=axes[1, 1], rot=45)
axes[1, 1].set_title('Contrast by Class')
axes[1, 1].set_xlabel('Animal Class')
axes[1, 1].set_ylabel('Contrast')
axes[1, 1].get_figure().suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# RGB color analysis by class
rgb_by_class = df_visual.groupby('class')[['red_mean', 'green_mean', 'blue_mean']].mean()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(rgb_by_class))
width = 0.25

ax.bar(x - width, rgb_by_class['red_mean'], width, label='Red', color='red', alpha=0.7)
ax.bar(x, rgb_by_class['green_mean'], width, label='Green', color='green', alpha=0.7)
ax.bar(x + width, rgb_by_class['blue_mean'], width, label='Blue', color='blue', alpha=0.7)

ax.set_xlabel('Animal Class')
ax.set_ylabel('Mean Channel Value')
ax.set_title('Average RGB Channel Values by Class')
ax.set_xticks(x)
ax.set_xticklabels(rgb_by_class.index, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Average image per class (to visualize dominant colors/patterns)
def compute_average_image(class_name, max_samples=50):
    """Compute average image for a class"""
    class_path = DATASET_PATH / class_name
    image_files = [f for f in class_path.iterdir() 
                   if f.suffix.lower() in image_extensions]
    
    samples = np.random.choice(image_files, size=min(max_samples, len(image_files)), replace=False)
    
    target_size = (224, 224)
    images = []
    
    for img_path in samples:
        try:
            with Image.open(img_path) as img:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                img_resized = img.resize(target_size)
                images.append(np.array(img_resized))
        except:
            continue
    
    if images:
        avg_img = np.mean(images, axis=0).astype(np.uint8)
        return avg_img
    return None

print("Computing average images...")
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, class_name in enumerate(classes):
    avg_img = compute_average_image(class_name)
    if avg_img is not None:
        axes[i].imshow(avg_img)
        axes[i].set_title(class_name)
        axes[i].axis('off')

plt.suptitle('Average Image per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Texture and Complexity Analysis

In [ ]:
from scipy import ndimage

def calculate_texture_features(img_array):
    gray = np.mean(img_array, axis=2) if len(img_array.shape) == 3 else img_array
    
    variance = np.var(gray)
    
    edges_x = ndimage.sobel(gray, axis=0)
    edges_y = ndimage.sobel(gray, axis=1)
    edge_magnitude = np.sqrt(edges_x**2 + edges_y**2)
    edge_density = np.mean(edge_magnitude)
    
    return variance, edge_density

print("Analyzing texture and complexity...")
texture_data = []

for class_name in classes:
    class_path = DATASET_PATH / class_name
    image_files = [f for f in class_path.iterdir() 
                   if f.suffix.lower() in image_extensions]
    
    # Sample for efficiency
    sample_size = min(50, len(image_files))
    sample_files = np.random.choice(image_files, size=sample_size, replace=False)
    
    for img_path in sample_files:
        try:
            with Image.open(img_path) as img:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                img_array = np.array(img)
                variance, edge_density = calculate_texture_features(img_array)
                
                texture_data.append({
                    'class': class_name,
                    'variance': variance,
                    'edge_density': edge_density
                })
        except:
            continue

df_texture = pd.DataFrame(texture_data)
print(f"Analyzed {len(df_texture)} images for texture properties")

In [ ]:
# Texture summary statistics
texture_summary = df_texture.groupby('class').agg({
    'variance': ['mean', 'std'],
    'edge_density': ['mean', 'std']
}).round(2)

print("Texture Complexity Summary by Class:")
print(texture_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Variance by class
df_texture.boxplot(column='variance', by='class', ax=axes[0], rot=45)
axes[0].set_title('Image Variance by Class (Detail Complexity)')
axes[0].set_xlabel('Animal Class')
axes[0].set_ylabel('Variance')
axes[0].get_figure().suptitle('')

# Edge density by class
df_texture.boxplot(column='edge_density', by='class', ax=axes[1], rot=45)
axes[1].set_title('Edge Density by Class (Structural Complexity)')
axes[1].set_xlabel('Animal Class')
axes[1].set_ylabel('Edge Density')
axes[1].get_figure().suptitle('')

plt.tight_layout()
plt.show()

## Correlation and Relationship Analysis

In [ ]:
metadata_agg = df_metadata.groupby('class').agg({
    'width': 'mean',
    'height': 'mean',
    'aspect_ratio': 'mean',
    'file_size_kb': 'mean'
}).reset_index()

visual_agg = df_visual.groupby('class').agg({
    'brightness': 'mean',
    'contrast': 'mean',
    'saturation': 'mean'
}).reset_index()

texture_agg = df_texture.groupby('class').agg({
    'variance': 'mean',
    'edge_density': 'mean'
}).reset_index()

df_merged = metadata_agg.merge(visual_agg, on='class')
df_merged = df_merged.merge(texture_agg, on='class')

numeric_features = ['width', 'height', 'aspect_ratio', 'file_size_kb', 
                    'brightness', 'contrast', 'saturation', 
                    'variance', 'edge_density']

correlation_matrix = df_merged[numeric_features].corr()

# Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Image Features (Class-level Averages)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nCorrelation analysis based on {len(df_merged)} classes")

## Key Findings and Insights

In [ ]:
print("\nDATASET COMPOSITION")
print("-" * 70)
print(f"Total Images: {df_distribution['Count'].sum()}")
print(f"Number of Classes: {num_classes}")
print(f"Classes: {', '.join(classes)}")
print(f"Average images per class: {df_distribution['Count'].mean():.0f}")
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")

print("\nIMAGE DIMENSIONS")
print("-" * 70)
print(f"Average size: {df_metadata['width'].mean():.0f} × {df_metadata['height'].mean():.0f} pixels")
print(f"Width range: [{df_metadata['width'].min()}, {df_metadata['width'].max()}]")
print(f"Height range: [{df_metadata['height'].min()}, {df_metadata['height'].max()}]")
print(f"Average aspect ratio: {df_metadata['aspect_ratio'].mean():.2f}")

print("\nVISUAL PROPERTIES")
print("-" * 70)
print(f"Average brightness: {df_visual['brightness'].mean():.2f} (0-255 scale)")
print(f"Average contrast: {df_visual['contrast'].mean():.2f}")
print(f"Average saturation: {df_visual['saturation'].mean():.2f}")

print("\nTEXTURE & COMPLEXITY")
print("-" * 70)
print(f"Average variance: {df_texture['variance'].mean():.2f}")
print(f"Average edge density: {df_texture['edge_density'].mean():.2f}")

print("\nSTORAGE")
print("-" * 70)
print(f"Total dataset size: {df_metadata['file_size_kb'].sum() / 1024:.2f} MB")
print(f"Average file size: {df_metadata['file_size_kb'].mean():.2f} KB")

print("\nCHALLENGES IDENTIFIED")
print("-" * 70)
challenges = []

if imbalance_ratio > 2:
    challenges.append(f"• Class imbalance ({imbalance_ratio:.1f}:1) - may require balancing techniques")

if df_metadata['width'].std() > df_metadata['width'].mean() * 0.5:
    challenges.append("• High variance in image dimensions - preprocessing required")

if len(non_rgb) > 0:
    challenges.append(f"• {len(non_rgb)} non-RGB images found - need conversion")

variance_cv = df_visual['brightness'].std() / df_visual['brightness'].mean()
if variance_cv > 0.3:
    challenges.append("• Significant brightness variation across dataset")

if len(challenges) == 0:
    print("• No major issues detected - dataset is well-structured")
else:
    for challenge in challenges:
        print(challenge)


## Class-Specific Analysis

In [ ]:
class_stats = pd.DataFrame()

for class_name in classes:
    class_meta = df_metadata[df_metadata['class'] == class_name]
    class_vis = df_visual[df_visual['class'] == class_name]
    class_tex = df_texture[df_texture['class'] == class_name]
    
    stats = {
        'Class': class_name,
        'Count': len(class_meta),
        'Avg Width': class_meta['width'].mean(),
        'Avg Height': class_meta['height'].mean(),
        'Brightness': class_vis['brightness'].mean(),
        'Contrast': class_vis['contrast'].mean(),
        'Variance': class_tex['variance'].mean(),
        'Edge Density': class_tex['edge_density'].mean()
    }
    
    class_stats = pd.concat([class_stats, pd.DataFrame([stats])], ignore_index=True)

class_stats = class_stats.round(2)
print("\nDetailed Statistics per Class:")
print(class_stats.to_string(index=False))

In [ ]:
print("\n" + "=" * 70)
print("POTENTIAL CLASSIFICATION CHALLENGES")
print("=" * 70)

brightness_by_class = df_visual.groupby('class')['brightness'].mean().sort_values()
contrast_by_class = df_visual.groupby('class')['contrast'].mean().sort_values()

print("\nBrightness ranking (darkest to brightest):")
for i, (class_name, brightness) in enumerate(brightness_by_class.items(), 1):
    print(f"  {i}. {class_name}: {brightness:.2f}")

print("\nContrast ranking (lowest to highest):")
for i, (class_name, contrast) in enumerate(contrast_by_class.items(), 1):
    print(f"  {i}. {class_name}: {contrast:.2f}")

# Intra-class variance (diversity within class)
print("\nIntra-class diversity (brightness std - higher means more varied):")
brightness_std = df_visual.groupby('class')['brightness'].std().sort_values(ascending=False)
for i, (class_name, std) in enumerate(brightness_std.items(), 1):
    diversity = "High" if std > df_visual['brightness'].std() else "Moderate" if std > df_visual['brightness'].std() * 0.7 else "Low"
    print(f"  {i}. {class_name}: {std:.2f} ({diversity})")

print("=" * 70)

## Extreme Cases Visualization

In [ ]:
def get_image_path_by_index(df, idx):
    """Retrieve image path from metadata"""
    class_name = df.iloc[idx]['class']
    class_path = DATASET_PATH / class_name
    image_files = list(class_path.glob('*'))
    if image_files:
        return image_files[0]  # Simplified: return first match
    return None

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

extremes = [
    ('Brightest', df_visual['brightness'].idxmax()),
    ('Darkest', df_visual['brightness'].idxmin()),
    ('Highest Contrast', df_visual['contrast'].idxmax()),
    ('Lowest Contrast', df_visual['contrast'].idxmin()),
    ('Most Complex', df_texture['variance'].idxmax()),
    ('Least Complex', df_texture['variance'].idxmin()),
    ('Largest', df_metadata['width'].idxmax()),
    ('Smallest', df_metadata['width'].idxmin())
]

for ax, (label, idx) in zip(axes.flat, extremes):
    class_name = None
    img_to_show = None
    
    if label in ['Brightest', 'Darkest', 'Highest Contrast', 'Lowest Contrast']:
        if idx < len(df_visual):
            class_name = df_visual.iloc[idx]['class']
    elif label in ['Most Complex', 'Least Complex']:
        if idx < len(df_texture):
            class_name = df_texture.iloc[idx]['class']
    else:
        if idx < len(df_metadata):
            class_name = df_metadata.iloc[idx]['class']
    
    if class_name:
        class_path = DATASET_PATH / class_name
        image_files = [f for f in class_path.iterdir() if f.suffix.lower() in image_extensions]
        if image_files:
            img_to_show = np.random.choice(image_files)
    
    if img_to_show:
        try:
            img = Image.open(img_to_show)
            ax.imshow(img)
            ax.set_title(f"{label}\n({class_name})", fontsize=10, fontweight='bold')
            ax.axis('off')
        except:
            ax.text(0.5, 0.5, 'Error loading', ha='center', va='center')
            ax.axis('off')
    else:
        ax.axis('off')

plt.suptitle('Extreme Cases in Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import os
os.makedirs('CSV', exist_ok=True)

class_stats.to_csv('CSV/phase1_class_statistics.csv', index=False)
df_distribution.to_csv('CSV/phase1_class_distribution.csv', index=False)